In [ ]:
import json
import random
from collections import Counter

from scipy.optimize import linear_sum_assignment

In [ ]:
DATASET = "vk-lsvd"
MATCHING_METHOD = "hungarian"  # "greedy" or "hungarian"

CODEBOOK_SIZE = 512
NUM_CODEBOOKS = 4

OLD_TRAIN_SPLIT = "0-8TR_9-10TE"
OLD_TRAIN_PART = "0-8TR"

NEW_TRAIN_SPLIT = "0-9TR_9-10TE"
NEW_TRAIN_PART = "0-8TR"


RESULTS_DIR = f"../temp_files/results_{DATASET}"

In [ ]:
old_train_semantics_path = (
    f"{RESULTS_DIR}/{OLD_TRAIN_SPLIT}/rqvae-collab/only_{OLD_TRAIN_PART}_clusters_colisionless.json"
)
old_all_train_semantics_path = f"{RESULTS_DIR}/{OLD_TRAIN_SPLIT}/rqvae-collab/all_clusters_colisionless.json"

new_train_semantics_path = (
    f"{RESULTS_DIR}/{NEW_TRAIN_SPLIT}/rqvae-collab/only_{NEW_TRAIN_PART}_clusters_colisionless.json"
)
new_all_train_semantics_path = f"{RESULTS_DIR}/{NEW_TRAIN_SPLIT}/rqvae-collab/all_clusters_colisionless.json"


with open(old_train_semantics_path) as f:
    old_train_semantics = json.load(f)
with open(old_all_train_semantics_path) as f:
    old_all_train_semantics = json.load(f)
with open(new_train_semantics_path) as f:
    new_train_semantics = json.load(f)
with open(new_all_train_semantics_path) as f:
    new_all_train_semantics = json.load(f)

In [ ]:
existing_new_ids = set()
existing_old_ids = set()
semantic_transitions = Counter()

common_item_ids = set(old_all_train_semantics.keys()) & set(new_all_train_semantics.keys())

for item_id in common_item_ids:
    old_semantic = old_all_train_semantics[item_id]
    new_semantic = new_all_train_semantics[item_id]

    assert len(old_semantic) == len(new_semantic) == NUM_CODEBOOKS

    existing_new_ids.update(new_semantic)
    existing_old_ids.update(old_semantic)

    for i in range(len(old_semantic)):
        state = (new_semantic[i], old_semantic[i])
        semantic_transitions[state] += 1

In [ ]:
if MATCHING_METHOD == "greedy":
    available_new_ids = set(existing_new_ids)
    available_old_ids = set(existing_old_ids)

    new_to_old_mapping = {}
    used_old_ids = set()

    for (new_id, old_id), score in sorted(semantic_transitions.items(), key=lambda x: -x[1]):
        if (new_id in available_new_ids) and (old_id in available_old_ids):
            new_to_old_mapping[new_id] = old_id
            available_new_ids.remove(new_id)
            available_old_ids.remove(old_id)
            used_old_ids.add(old_id)

    existing_new_ids = sorted(available_new_ids)
    existing_old_ids = sorted(available_old_ids)

elif MATCHING_METHOD == "hungarian":
    new_ids = sorted(existing_new_ids)
    old_ids = sorted(existing_old_ids)

    new2i = {sid: i for i, sid in enumerate(new_ids)}
    old2j = {sid: j for j, sid in enumerate(old_ids)}

    W = [[0.0 for _ in range(len(old_ids))] for _ in range(len(new_ids))]

    for (new_id, old_id), c in semantic_transitions.items():
        i = new2i.get(new_id)
        j = old2j.get(old_id)
        if i is None or j is None:
            continue
        W[i][j] = float(c)

    row_indices, col_indices = linear_sum_assignment(W, maximize=True)

    new_to_old_mapping = {}
    for i, j in zip(row_indices, col_indices, strict=True):
        new_to_old_mapping[new_ids[i]] = old_ids[j]

    matched_new = set(new_to_old_mapping.keys())
    matched_old = set(new_to_old_mapping.values())
    existing_new_ids = sorted(set(new_ids) - matched_new)
    existing_old_ids = sorted(set(old_ids) - matched_old)
else:
    raise ValueError("MATCHING_METHOD must be 'greedy' or 'hungarian'")

In [ ]:
existing_old_ids = sorted(existing_old_ids)
existing_new_ids = sorted(existing_new_ids)

for i in range(max(len(existing_old_ids), len(existing_new_ids))):
    old_id = existing_old_ids[i] if i < len(existing_old_ids) else -1
    new_id = existing_new_ids[i] if i < len(existing_new_ids) else -1

    while old_id == -1:
        tmp_id = random.randint(
            (new_id // CODEBOOK_SIZE) * CODEBOOK_SIZE, (new_id // CODEBOOK_SIZE + 1) * CODEBOOK_SIZE - 1
        )
        if tmp_id not in existing_new_ids:
            old_id = tmp_id

    while new_id == -1:
        tmp_id = random.randint(
            (old_id // CODEBOOK_SIZE) * CODEBOOK_SIZE, (old_id // CODEBOOK_SIZE + 1) * CODEBOOK_SIZE - 1
        )
        if tmp_id not in existing_old_ids:
            new_id = tmp_id

    assert old_id // CODEBOOK_SIZE == new_id // CODEBOOK_SIZE
    new_to_old_mapping[new_id] = old_id

In [ ]:
old_train_semantics_changed = {}
old_all_semantics_changed = {}

for item_id, semantic in new_train_semantics.items():
    old_train_semantics_changed[item_id] = [new_to_old_mapping[x] for x in semantic]

for item_id, semantic in new_all_train_semantics.items():
    old_all_semantics_changed[item_id] = [new_to_old_mapping[x] for x in semantic]

In [ ]:
updated_train_semantics_path = (
    f"{RESULTS_DIR}/{NEW_TRAIN_SPLIT}/rqvae-collab/{MATCHING_METHOD}_"
    f"to_{OLD_TRAIN_SPLIT}_only_{NEW_TRAIN_PART}_clusters_colisionless.json"
)
updated_all_train_semantics_path = (
    f"{RESULTS_DIR}/{NEW_TRAIN_SPLIT}/rqvae-collab/{MATCHING_METHOD}_"
    f"to_{OLD_TRAIN_SPLIT}_all_clusters_colisionless.json"
)

with open(updated_train_semantics_path, "w") as f:
    json.dump(old_train_semantics_changed, f, indent=2)
with open(updated_all_train_semantics_path, "w") as f:
    json.dump(old_all_semantics_changed, f, indent=2)